In [2]:
import pandas as pd
import numpy as np


def production_estimate(df,OP, EQ, PROD, CAP):
    df = df[df['capacity_ttpa']!=0]
    df.loc[df['capacity_ttpa'] < df['production_ttpa'], 'capacity_ttpa'] = df['production_ttpa']
    df = df.copy()
    df[PROD] = pd.to_numeric(df[PROD], errors='coerce').fillna(0)
    df[CAP]  = pd.to_numeric(df[CAP],  errors='coerce')

    # actual ratio only where well-defined: a producing plant with real capacity
    producing = df[PROD] > 0  # production is non-zero
    df['_ratio'] = np.where(producing, df[PROD] / df[CAP], np.nan)

    # operator-own, same-equipment average ratio (producing plants only)
    # NaN => this operator/equipment group has NO producing plant
    grp_avg = df.groupby([OP, EQ])['_ratio'].transform('mean')
    has_nonzero = grp_avg.notna()

    # Estimate utilization rate
    df['utilization_rate'] = np.where(
        df[PROD] > 0, df['_ratio'],            # producing -> own ratio
        np.where(has_nonzero, grp_avg, 0.0))          # idle -> group avg, else 0
    # Impute production
    df['production_ttpa_est'] = np.where(
        df[PROD] > 0, df[PROD],                        # keep actual
        np.where(has_nonzero, grp_avg * df[CAP], 0.0)) # impute, else 0

    return df.drop(columns = '_ratio')


steel_plant_df = pd.read_excel('../data/Processed_data/steel_property_cost_merged.xlsx', sheet_name='Sheet1')
OP, EQ, PROD, CAP = 'Operator', 'Main production equipment', 'production_ttpa', 'capacity_ttpa'
new_df = production_estimate(steel_plant_df, OP, EQ, PROD, CAP)

In [7]:
new_df.to_excel('../data/Processed_data/test.xlsx', index=False)

In [10]:
# test the steel plant data
total_production = steel_plant_df[
    steel_plant_df['Main production equipment'].isin(['EAF', 'BOF', 'Steel other/unspecified'])
    & (steel_plant_df['Status'] == 'operating')
]['production_ttpa'].sum()

print(total_production/1800000)

0.5174677777777777
